# Phase 2 — Feature Extraction

## Goal
Convert the cleaned text into numbers (token IDs) that BART can understand.

## Algorithm: BPE (Byte Pair Encoding)
The BART tokenizer uses BPE — it splits words into frequent subword units.
- "learning" → ["learn", "ing"]
- "deep learning" → [1996, 4083]

This handles unknown words and reduces vocabulary size.

In [1]:
import os
os.environ["HF_HOME"] = r"D:\hf_cache"

from datasets import load_from_disk
from transformers import AutoTokenizer

D:\4thGrade_01\Term02\ANLP\ResearchPaperSummarization(PDFs)_ANLP_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load Processed Dataset

In [2]:
train_data = load_from_disk("../data/processed/train")
test_data  = load_from_disk("../data/processed/test")

print(f"Train: {len(train_data)} examples")
print(f"Test:  {len(test_data)} examples")

Train: 1600 examples
Test:  400 examples


## 2. Load Tokenizer

In [3]:
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")

# Quick test
sample_text = "Deep learning is a subset of machine learning."
tokens = tokenizer(sample_text)

print(f"Original text:    {sample_text}")
print(f"Token IDs:        {tokens['input_ids']}")
print(f"Number of tokens: {len(tokens['input_ids'])}")

Original text:    Deep learning is a subset of machine learning.
Token IDs:        [0, 35166, 2239, 16, 10, 37105, 9, 3563, 2239, 4, 2]
Number of tokens: 11


## 3. Tokenize Dataset

Each example is converted to:
- `input_ids` → integer IDs for each token in the document (512 tokens)
- `attention_mask` → 1 for real tokens, 0 for padding
- `labels` → integer IDs for the summary (128 tokens)

`truncation=True` → cut if longer than max length
`padding="max_length"` → pad with zeros if shorter (GPU requires same-size inputs)

In [4]:
MAX_INPUT_LENGTH  = 512
MAX_TARGET_LENGTH = 128

def tokenize_sample(sample):
    model_inputs = tokenizer(
        sample["document"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )
    labels = tokenizer(
        sample["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# batched=True → processes multiple examples at once (faster)
train_tokenized = train_data.map(tokenize_sample, batched=True)
test_tokenized  = test_data.map(tokenize_sample,  batched=True)

print("Tokenization done!")

Map: 100%|██████████| 400/400 [00:00<00:00, 2457.86 examples/s]

Tokenization done!


## 4. Verify Output

In [5]:
print("--- Verification ---")
print(f"input_ids length:      {len(train_tokenized[0]['input_ids'])}")
print(f"attention_mask length: {len(train_tokenized[0]['attention_mask'])}")
print(f"labels length:         {len(train_tokenized[0]['labels'])}")

--- Verification ---
input_ids length:      512
attention_mask length: 512
labels length:         128


## 5. Save Tokenized Dataset

In [6]:
os.makedirs("../data/processed", exist_ok=True)
train_tokenized.save_to_disk("../data/processed/train_tokenized")
test_tokenized.save_to_disk("../data/processed/test_tokenized")

print("Tokenized data saved!")
print("Phase 2 complete!")

Saving the dataset (1/1 shards): 100%|██████████| 400/400 [00:00<00:00, 63095.96 examples/s]

Tokenized data saved!
Phase 2 complete!
